# ห้องทดลอง metric ความชัน — `slope_metrics.py`

โน้ตบุ๊กนี้คือเวอร์ชันรันทีละส่วนของ `python slope_metrics.py` โดยเปลี่ยนผลจาก
**บรรทัด print** เป็น **ตาราง** และเพิ่มการแบ่งผลตาม **รูปแบบถนน** (normal / curve / night)

### สูตรอยู่ที่ไหน

| ไฟล์ | หน้าที่ |
| --- | --- |
| `slope_metrics.py` | **สูตรตัวจริง** แก้ที่นี่ที่เดียวเท่านั้น |
| `slope_metrics_lab.ipynb` (ไฟล์นี้) | ทดสอบ + แสดงตาราง — `import` สูตรมาใช้ ไม่มีสำเนาโค้ด |

โน้ตบุ๊กนี้ **ไม่มีการนิยามฟังก์ชันวัดผลซ้ำ** ถ้าอยากแก้พฤติกรรมให้ไปแก้ `.py`
แล้ว restart kernel เซลล์ในนี้จะเห็นของใหม่ทันที (CLAUDE.md กำหนดไว้แบบนี้เพื่อไม่ให้มีสองที่ให้แก้)

### ไฟล์นี้ต่างจาก `evaluate_metrics_comparison.ipynb` ยังไง

สองไฟล์ตอบคนละคำถาม อย่าสับสน

| | คำถามที่ตอบ | ใช้โมเดลไหม |
| --- | --- | --- |
| **ไฟล์นี้** | *"ตัว metric เชื่อได้ไหม และมันลำเอียงตามรูปแบบถนนหรือเปล่า"* | ❌ ไม่ใช้ — บิดเบือน "เฉลย" ด้วยจำนวนที่เรารู้คำตอบอยู่แล้ว |
| `evaluate_metrics_comparison.ipynb` | *"โมเดลเก่งแค่ไหน และ IoU กับความชันให้คำตอบต่างกันตรงไหน"* | ✅ ใช้ ต้องมี GPU |

ไฟล์นี้ไม่ใช้ GPU ไม่ใช้ `torch` / `smp` และรันจบใน **~2 นาที**


In [1]:
# ถ้ายังไม่มีของพวกนี้ ให้เอา # ออกแล้วรันหนึ่งครั้ง
# ต้องใส่ --user ไม่งั้นของจะหายเมื่อ container restart (CLAUDE.md)
# !pip install --user opencv-python numpy scipy pandas


In [2]:
import os
import sys
import csv
import random
import inspect
import collections

import numpy as np
import cv2
cv2.setNumThreads(0)      # เครื่องนี้มี 4 cores จริง อย่าให้ cv2 แตก thread แย่ง CPU
import pandas as pd

sys.path.append('.')      # ให้ import slope_metrics จาก project root ได้

# ---- import สูตรทั้งหมดจากไฟล์ตัวจริง ไม่มีสำเนาในโน้ตบุ๊กนี้ ----
from slope_metrics import (
    # ค่าคงที่
    SLOPE_MIN_AREA, SLOPE_MIN_ROWS, SLOPE_MATCH_GATE, SLOPE_MAX_RMS, SLOPE_Y_REF_FRAC,
    DEFAULT_ROOT,
    # สูตรหลัก
    extract_lanes, angle_error, match_lanes,
    slope_metrics_for_pair, slope_metrics_for_lanes, mask_iou,
    AnglePositionPrior,
    # ฟังก์ชันบิดเบือน (ใช้ทดสอบว่า metric ตัวไหนเห็นอะไร)
    shift_mask, dilate_mask, erode_mask, rotate_lanes, drop_one_lane,
    # ตัวช่วยวาดเส้นตรงที่รู้มุมจริง ใช้ใน self-check
    _draw_line_mask,
)

DATASET_ROOT = DEFAULT_ROOT
MASK_DIR = os.path.join(DATASET_ROOT, "masks")
TEST_CSV = os.path.join(DATASET_ROOT, "test_list.csv")

# ==================== ตั้งค่า ====================
N_PER_CAT  = 400    # จำนวนภาพ "ต่อหมวด" ในตารางหลัก (สมดุลกัน ไม่ใช่ตามสัดส่วนจริง)
N_NULL_CAT = 100    # จำนวนภาพต่อหมวดสำหรับ null test (เป็นด่านตรวจ ใช้น้อยกว่าได้)
SEED       = 11
# =================================================

CATS = ("normal", "curve", "night")

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 60)

print(f"✅ import สูตรจาก slope_metrics.py แล้ว ({inspect.getsourcefile(extract_lanes)})")
print(f"   dataset: {DATASET_ROOT}")
print(f"   มี test_list.csv: {os.path.exists(TEST_CSV)}")


✅ import สูตรจาก slope_metrics.py แล้ว (/home/jovyan/cardashcam_train_model/slope_metrics.py)
   dataset: /home/jovyan/shared/donut/CarDashCam/CULane
   มี test_list.csv: True


## 1. อ่านโค้ดจริงในโน้ตบุ๊ก (ไม่ใช่สำเนา)

`inspect.getsource()` ดึงโค้ด **ตัวจริงที่กำลังทำงานอยู่** ออกมาจาก `slope_metrics.py`
ถ้าไฟล์เปลี่ยน สิ่งที่แสดงตรงนี้เปลี่ยนตามทันที จึงไม่มีทางหลุด sync แบบก๊อปโค้ดมาวาง


In [3]:
def show_source(fn):
    """แสดงโค้ดตัวจริงจาก slope_metrics.py (ไม่ใช่สำเนา)"""
    src = inspect.getsource(fn)
    n = len(src.splitlines())
    print(f"# ---- {fn.__name__}  ({n} บรรทัด จาก {os.path.basename(inspect.getsourcefile(fn))}) ----")
    print(src)


# สารบัญฟังก์ชันทั้งหมดที่โมดูลนี้ให้มา
_FUNCS = [
    ("extract_lanes",          extract_lanes,          "แยก mask เป็นรายเส้น + หามุมของแต่ละเส้น"),
    ("angle_error",            angle_error,            "ผลต่างมุมสองเส้น พับเข้าช่วง [0,90]"),
    ("match_lanes",            match_lanes,            "จับคู่เส้น GT กับ pred (Hungarian) ด้วยตำแหน่ง x"),
    ("slope_metrics_for_pair", slope_metrics_for_pair, "รับ mask สองอัน คืนผลการจับคู่ + error"),
    ("slope_metrics_for_lanes", slope_metrics_for_lanes, "เหมือนข้างบน แต่รับเส้นที่แยกมาแล้ว (เร็วกว่า)"),
    ("mask_iou",               mask_iou,               "IoU ต่อภาพ บน numpy mask"),
    ("shift_mask",             shift_mask,             "บิดเบือน: เลื่อนทั้ง mask ไปด้านข้าง"),
    ("dilate_mask",            dilate_mask,            "บิดเบือน: ทำเส้นหนาขึ้น"),
    ("erode_mask",             erode_mask,             "บิดเบือน: ทำเส้นบางลง"),
    ("rotate_lanes",           rotate_lanes,           "บิดเบือน: หมุนแต่ละเส้นรอบ centroid ของตัวเอง"),
    ("drop_one_lane",          drop_one_lane,          "บิดเบือน: ลบเส้นทิ้งหนึ่งเส้น"),
    ("AnglePositionPrior",     AnglePositionPrior,     "baseline ไม่เรียนรู้: เดามุมจากตำแหน่ง x อย่างเดียว"),
]

toc = pd.DataFrame(
    [(n, len(inspect.getsource(f).splitlines()), d) for n, f, d in _FUNCS],
    columns=["ฟังก์ชัน / คลาส", "บรรทัด", "หน้าที่"],
).set_index("ฟังก์ชัน / คลาส")
print("สารบัญของทุกอย่างที่ slope_metrics.py ให้มา (ครบทั้งโมดูล) — เปลี่ยนชื่อในเซลล์ถัดไปเพื่อดูโค้ดตัวอื่น\n")
toc


สารบัญของทุกอย่างที่ slope_metrics.py ให้มา (ครบทั้งโมดูล) — เปลี่ยนชื่อในเซลล์ถัดไปเพื่อดูโค้ดตัวอื่น



,บรรทัด,หน้าที่
ฟังก์ชัน / คลาส,,
extract_lanes,55,แยก mask เป็นรายเส้น + หามุมของแต่ละเส้น
angle_error,7,"ผลต่างมุมสองเส้น พับเข้าช่วง [0,90]"
match_lanes,13,จับคู่เส้น GT กับ pred (Hungarian) ด้วยตำแหน่ง x
slope_metrics_for_pair,8,รับ mask สองอัน คืนผลการจับคู่ + error
slope_metrics_for_lanes,12,เหมือนข้างบน แต่รับเส้นที่แยกมาแล้ว (เร็วกว่า)
mask_iou,7,IoU ต่อภาพ บน numpy mask
shift_mask,10,บิดเบือน: เลื่อนทั้ง mask ไปด้านข้าง
dilate_mask,4,บิดเบือน: ทำเส้นหนาขึ้น
erode_mask,4,บิดเบือน: ทำเส้นบางลง


In [4]:
show_source(extract_lanes)       # <-- เปลี่ยนชื่อตรงนี้เพื่อดูฟังก์ชันอื่น เช่น show_source(match_lanes)


# ---- extract_lanes  (55 บรรทัด จาก slope_metrics.py) ----
def extract_lanes(binmask, min_area=SLOPE_MIN_AREA, min_rows=SLOPE_MIN_ROWS,
                  y_ref_frac=SLOPE_Y_REF_FRAC):
    """
    แยกเส้นเลนแต่ละเส้นออกจาก mask ไบนารี แล้วหาความชันของแต่ละเส้น

    คืน list ของ dict เรียงจากซ้ายไปขวา แต่ละตัวมี
      a     : ความชัน dx/dy ของเส้น (x = a*y + b)
      b     : จุดตัด
      ang   : มุมองศาในช่วง [0,180) โดย 90 = เส้นตั้งฉาก
      rms   : ค่าความคลาดของ fit (px) ยิ่งน้อยยิ่งเป็นเส้นตรง
      xref  : ตำแหน่ง x ที่ y = y_ref_frac*H ใช้สำหรับจับคู่เส้น
      area, nrows, y_top, y_bot

    mask ว่างเปล่าจะคืน list ว่าง (ไม่ error)

    ทำไมไม่ใช้ PCA: เส้นเลนใน mask เป็นแถบหนา ถ้าเอา PCA กับกลุ่มพิกเซลตรงๆ
    ความหนาและความโค้งจะดึงแกนหลักเพี้ยน (ทดลองแล้วได้มุม 5 องศาทั้งที่เป็นเส้นแนวตั้ง)
    การยุบเป็น centerline ก่อนแก้ปัญหานี้
    """
    binmask = np.asarray(binmask)
    if binmask.ndim != 2:
        binmask = binmask.reshape(binmask.shape[-2], binmask.shape[-1])
    bi

## 2. ค่าคงที่ควบคุมพฤติกรรม

ห้าตัวนี้กำหนดว่าอะไรนับเป็น "เส้น" และอะไรนับเป็น "คู่ที่จับได้"
ค่าที่ตั้งไว้มาจากการวัดจริง ไม่ใช่เดา — คอลัมน์สุดท้ายบอกว่าถ้าขยับแล้วจะพังทางไหน


In [5]:
const_tbl = pd.DataFrame([
    ("SLOPE_MIN_AREA",   SLOPE_MIN_AREA,   "px²",
     "พื้นที่ต่ำสุดที่นับเป็นเส้น",
     "ลด = รับ noise เข้ามาเป็นเส้นปลอม / เพิ่ม = เส้นบางไกลๆ หลุดหาย"),
    ("SLOPE_MIN_ROWS",   SLOPE_MIN_ROWS,   "แถว",
     "จำนวนแถวขั้นต่ำที่ต้องมีจุด ถึงจะ fit เส้นตรงได้",
     "ลด = fit จากจุดน้อยเกิน มุมแกว่ง / เพิ่ม = เส้นสั้นถูกทิ้ง"),
    ("SLOPE_MATCH_GATE", SLOPE_MATCH_GATE, "px",
     "ระยะ x สูงสุดที่ยอมให้จับคู่ GT กับ pred",
     "ลด = จับคู่ไม่ติด recall ตก / เพิ่ม = จับข้ามฝั่งถนน มุมคลาดพุ่ง"),
    ("SLOPE_MAX_RMS",    SLOPE_MAX_RMS,    "px",
     "เกินนี้ถือว่าเส้นนั้น 'ความชันนิยามไม่ได้' (โค้งจัด/เลนติดกัน)",
     "ใช้แยกคอลัมน์ *_wf ออกมา ไม่ได้กรองเส้นทิ้ง"),
    ("SLOPE_Y_REF_FRAC", SLOPE_Y_REF_FRAC, "สัดส่วนความสูง",
     "จับคู่ด้วยตำแหน่ง x ที่ความสูงนี้ (0.9 = ใกล้ตัวรถ)",
     "ใกล้ 0 = ใช้ปลายเส้นไกลๆ ที่เลนบีบเข้าหากัน แยกกันไม่ออก"),
], columns=["ค่าคงที่", "ค่า", "หน่วย", "หน้าที่", "ถ้าเปลี่ยนแล้วเกิดอะไร"]).set_index("ค่าคงที่")
const_tbl


,ค่า,หน่วย,หน้าที่,ถ้าเปลี่ยนแล้วเกิดอะไร
ค่าคงที่,,,,
SLOPE_MIN_AREA,200.0,px²,พื้นที่ต่ำสุดที่นับเป็นเส้น,ลด = รับ noise เข้ามาเป็นเส้นปลอม / เพิ่ม = เส้นบางไกลๆ ...
SLOPE_MIN_ROWS,12.0,แถว,จำนวนแถวขั้นต่ำที่ต้องมีจุด ถึงจะ fit เส้นตรงได้,ลด = fit จากจุดน้อยเกิน มุมแกว่ง / เพิ่ม = เส้นสั้นถูกทิ้ง
SLOPE_MATCH_GATE,150.0,px,ระยะ x สูงสุดที่ยอมให้จับคู่ GT กับ pred,ลด = จับคู่ไม่ติด recall ตก / เพิ่ม = จับข้ามฝั่งถนน มุม...
SLOPE_MAX_RMS,10.0,px,เกินนี้ถือว่าเส้นนั้น 'ความชันนิยามไม่ได้' (โค้งจัด/เลนต...,ใช้แยกคอลัมน์ *_wf ออกมา ไม่ได้กรองเส้นทิ้ง
SLOPE_Y_REF_FRAC,0.9,สัดส่วนความสูง,จับคู่ด้วยตำแหน่ง x ที่ความสูงนี้ (0.9 = ใกล้ตัวรถ),ใกล้ 0 = ใช้ปลายเส้นไกลๆ ที่เลนบีบเข้าหากัน แยกกันไม่ออก


## 3. self-check ที่ไม่ต้องใช้ dataset

สามหัวข้อแรก ตรงกับ `_selftest_synthetic()` ใน `slope_metrics.py`
ต่างกันแค่แสดงเป็นตารางแทนบรรทัด print

**ทั้งสามหัวข้อต้องผ่านหมด** ถ้าไม่ผ่านแปลว่าสูตรมีบั๊ก อย่าอ่านผลในหัวข้อถัดๆ ไป


In [6]:
# ---- 3.1 มุมสังเคราะห์: วาดเส้นตรงที่รู้มุมจริง แล้วดูว่า extract_lanes อ่านกลับมาถูกไหม ----
TOL_DEG = 1.5

rows = []
for target in (30.0, 45.0, 90.0, 135.0, 150.0):
    lanes = extract_lanes(_draw_line_mask(target))
    if len(lanes) != 1:
        rows.append((f"{target:.0f}°", np.nan, np.nan, np.nan, len(lanes), "✗ เจอเส้นผิดจำนวน"))
        continue
    got = lanes[0]["ang"]
    err = angle_error(got, target)
    rows.append((f"{target:.0f}°", round(got, 2), round(err, 3),
                 round(lanes[0]["rms"], 2), 1, "✓" if err <= TOL_DEG else "✗"))

t31 = pd.DataFrame(rows, columns=["มุมที่วาด", "มุมที่วัดได้", "คลาด (°)",
                                  "rms (px)", "จำนวนเส้นที่เจอ", "ผล"]).set_index("มุมที่วาด")
pass_31 = (t31["ผล"] == "✓").all()
print(f"3.1 มุมสังเคราะห์ (ยอมคลาดได้ ≤ {TOL_DEG}°) — {'✅ ผ่าน' if pass_31 else '❌ ไม่ผ่าน'}\n")
t31


3.1 มุมสังเคราะห์ (ยอมคลาดได้ ≤ 1.5°) — ✅ ผ่าน



,มุมที่วัดได้,คลาด (°),rms (px),จำนวนเส้นที่เจอ,ผล
มุมที่วาด,,,,,
30°,30.06,0.058,1.23,1,✓
45°,45.06,0.062,0.69,1,✓
90°,90.00,0.000,0.00,1,✓
135°,134.94,0.062,0.69,1,✓
150°,149.94,0.058,1.23,1,✓


In [7]:
# ---- 3.2 การพับมุม: เส้นตรงไม่มีทิศทาง 1° กับ 179° ต่างกันแค่ 2° ไม่ใช่ 178° ----
rows = []
for a1, a2, expect in ((1.0, 179.0, 2.0), (10.0, 170.0, 20.0),
                       (0.0, 90.0, 90.0), (89.0, 91.0, 2.0),
                       (45.0, 135.0, 90.0), (170.0, 10.0, 20.0)):
    got = angle_error(a1, a2)
    ok = abs(got - expect) < 1e-6
    rows.append((f"{a1:.0f}° vs {a2:.0f}°", expect, round(got, 4), "✓" if ok else "✗"))

t32 = pd.DataFrame(rows, columns=["คู่มุม", "ควรได้ (°)", "ได้ (°)", "ผล"]).set_index("คู่มุม")
pass_32 = (t32["ผล"] == "✓").all()
print(f"3.2 การพับมุมเข้าช่วง [0, 90] — {'✅ ผ่าน' if pass_32 else '❌ ไม่ผ่าน'}\n")
t32


3.2 การพับมุมเข้าช่วง [0, 90] — ✅ ผ่าน



,ควรได้ (°),ได้ (°),ผล
คู่มุม,,,
1° vs 179°,2.0,2.0,✓
10° vs 170°,20.0,20.0,✓
0° vs 90°,90.0,90.0,✓
89° vs 91°,2.0,2.0,✓
45° vs 135°,90.0,90.0,✓
170° vs 10°,20.0,20.0,✓


In [8]:
# ---- 3.3 เคสขอบ: input แปลกๆ ต้องไม่ crash และต้องคืนค่าว่างอย่างถูกต้อง ----
_tiny = np.pad(np.ones((4, 4), np.uint8), ((10, 434), (10, 786)))   # จุดจิ๋ว area = 16 px

_g2 = [dict(xref=100.0, ang=80.0, rms=1.0), dict(xref=600.0, ang=100.0, rms=1.0)]
_p3 = [dict(xref=105.0, ang=82.0, rms=1.0), dict(xref=610.0, ang=99.0, rms=1.0),
       dict(xref=400.0, ang=90.0, rms=1.0)]

checks = [
    ("mask ว่างเปล่า",              len(extract_lanes(np.zeros((448, 800), np.uint8))), 0,
     "ไม่มีอะไรใน mask ต้องได้ 0 เส้น"),
    (f"จุดจิ๋ว area=16 < {SLOPE_MIN_AREA}", len(extract_lanes(_tiny)),                  0,
     "เล็กกว่า SLOPE_MIN_AREA ต้องถูกกรองทิ้ง"),
    ("match: GT ว่าง",              len(match_lanes([], [dict(xref=1.0)])),             0,
     "ไม่มีเฉลย จับคู่ไม่ได้"),
    ("match: pred ว่าง",            len(match_lanes([dict(xref=1.0)], [])),             0,
     "ไม่มีคำทำนาย จับคู่ไม่ได้"),
    (f"match: ห่างเกิน gate {SLOPE_MATCH_GATE:.0f}px",
     len(match_lanes([dict(xref=0.0)], [dict(xref=500.0)])),                            0,
     "ห่างเกิน gate ต้องไม่จับคู่ข้ามฝั่งถนน"),
    ("match: GT 2 เส้น vs pred 3 เส้น", len(match_lanes(_g2, _p3)),                     2,
     "จับได้มากสุดเท่าฝั่งที่น้อยกว่า"),
]

t33 = pd.DataFrame(
    [(n, exp, got, "✓" if got == exp else "✗", why) for n, got, exp, why in checks],
    columns=["เคส", "ควรได้", "ได้", "ผล", "ทำไม"]).set_index("เคส")
pass_33 = (t33["ผล"] == "✓").all()
print(f"3.3 เคสขอบ — {'✅ ผ่าน' if pass_33 else '❌ ไม่ผ่าน'}\n")
t33


3.3 เคสขอบ — ✅ ผ่าน



,ควรได้,ได้,ผล,ทำไม
เคส,,,,
mask ว่างเปล่า,0,0,✓,ไม่มีอะไรใน mask ต้องได้ 0 เส้น
จุดจิ๋ว area=16 < 200,0,0,✓,เล็กกว่า SLOPE_MIN_AREA ต้องถูกกรองทิ้ง
match: GT ว่าง,0,0,✓,ไม่มีเฉลย จับคู่ไม่ได้
match: pred ว่าง,0,0,✓,ไม่มีคำทำนาย จับคู่ไม่ได้
match: ห่างเกิน gate 150px,0,0,✓,ห่างเกิน gate ต้องไม่จับคู่ข้ามฝั่งถนน
match: GT 2 เส้น vs pred 3 เส้น,2,2,✓,จับได้มากสุดเท่าฝั่งที่น้อยกว่า


## 4. null test บนข้อมูลจริง — **แยกตามรูปแบบถนน**

เอา "เฉลย" ใส่เป็น "คำทำนาย" ตรงๆ ผลที่ถูกต้องมีทางเดียวคือ

- มุมคลาด **สูงสุด** = `0.0000°` (ใช้ max ไม่ใช่ median เพราะเป็นด่านตรวจ ต้องเป๊ะทุกเส้น)
- IoU = `1.0000`
- จับคู่ได้ **ครบทุกเส้น**

ถ้าหมวดไหนไม่ผ่าน แปลว่าโค้ดวัดผลมีบั๊ก **ห้ามอ่านตารางในหัวข้อ 5 ต่อ**


In [9]:
def load_by_category(n_per_cat, seed=SEED):
    """
    สุ่มภาพ 'หมวดละเท่ากัน' จาก test split

    ทำไมต้องสมดุล: test split เอียงมาก (normal 8,338 / curve 2,407 / night 2,553)
    ถ้า stride ทั้งไฟล์แบบที่ไฟล์อื่นทำ จะได้ curve แค่ ~70 ภาพ กำลังทางสถิติไม่พอจะแยกหมวด
    """
    rows = list(csv.DictReader(open(TEST_CSV)))
    bycat = collections.defaultdict(list)
    for r in rows:
        bycat[r["category"]].append(r)
    out = {}
    for cat in CATS:
        sel = bycat[cat]
        step = max(1, len(sel) // n_per_cat)      # stride = กระจายทั้ง split ไม่กระจุกที่คลิปเดียว
        out[cat] = sel[::step][:n_per_cat]
    return out


def read_mask(rec):
    m = cv2.imread(os.path.join(MASK_DIR, rec["filename"].replace(".jpg", ".png")),
                   cv2.IMREAD_GRAYSCALE)
    return None if m is None else m > 127


# ---- null test ----
picks_null = load_by_category(N_NULL_CAT)
rows = []
for cat in CATS:
    errs, ious, n_gt, n_match, n_img = [], [], 0, 0, 0
    for rec in picks_null[cat]:
        gt = read_mask(rec)
        if gt is None:
            continue
        g = extract_lanes(gt)
        if not g:
            continue
        n_img += 1
        res = slope_metrics_for_lanes(g, g)
        n_gt += res["n_gt"]
        n_match += res["n_matched"]
        errs += [d["err"] for d in res["pairs"]]
        ious.append(mask_iou(gt, gt))
    max_err = float(np.max(errs)) if errs else float("nan")
    mean_iou = float(np.mean(ious)) if ious else float("nan")
    ok = max_err < 1e-9 and abs(mean_iou - 1.0) < 1e-6 and n_match == n_gt
    rows.append((cat, n_img, n_gt, f"{max_err:.4f}", f"{mean_iou:.4f}",
                 f"{n_match}/{n_gt}", "✓" if ok else "✗"))

t4 = pd.DataFrame(rows, columns=["ถนน", "ภาพ", "เส้น GT", "มุมคลาดสูงสุด (°)",
                                 "IoU", "จับคู่ได้", "ผล"]).set_index("ถนน")
pass_4 = (t4["ผล"] == "✓").all()
print(f"4. null test แยกหมวด — {'✅ ผ่านทุกหมวด' if pass_4 else '❌ ไม่ผ่าน — metric มีบั๊ก อย่าอ่านต่อ'}\n")
t4


4. null test แยกหมวด — ✅ ผ่านทุกหมวด



,ภาพ,เส้น GT,มุมคลาดสูงสุด (°),IoU,จับคู่ได้,ผล
ถนน,,,,,,
normal,100,303,0.0000,1.0000,303/303,✓
curve,100,275,0.0000,1.0000,275/275,✓
night,100,281,0.0000,1.0000,281/281,✓


## 5. ตารางหลัก — บิดเบือนเฉลย แยกตามรูปแบบถนน

วิธี: เอา **เฉลย** มาบิดเบือนด้วยจำนวนที่เรากำหนดเอง (จึงรู้คำตอบที่ถูกอยู่แล้ว)
แล้วดูว่า **IoU** กับ **มุมคลาด** ขยับไปแค่ไหน แยกตาม normal / curve / night

ทำแบบนี้เพราะจะเทียบ IoU `0.535` กับมุมคลาด `0.95°` ตรงๆ ไม่ได้ — คนละหน่วย
คำถามที่ตอบได้จริงคือ **"ความผิดพลาดแบบไหน metric ตัวไหนมองเห็น"**

**ด่านตรวจสองตัวในเซลล์นี้** ถ้าไม่ผ่านคือมีบั๊ก อย่าอ่านตัวเลขต่อ
1. แถว identity ต้องได้ IoU `1.000` และมุมคลาด `0.00°` เป๊ะทุกหมวด
2. แถวหมุน `d°` ต้องอ่านกลับมาได้ `~d°` ทุกหมวด (พิสูจน์ว่าฟังก์ชันบิดเบือนเองถูกต้อง)


In [10]:
def perturbation_by_category(n_per_cat=N_PER_CAT, seed=SEED, save_csv=True):
    """บิดเบือนเฉลยด้วยจำนวนที่รู้คำตอบ แล้วรายงาน IoU / มุมคลาด / recall แยกตามหมวดถนน"""
    rng = random.Random(seed)
    picks = load_by_category(n_per_cat, seed)

    # (ชื่อ, ฟังก์ชัน, มุมคลาดที่ควรได้ตามทฤษฎี)
    schemes = [
        ("ไม่ทำอะไร (identity)", lambda m: m,                      0.0),
        ("เลื่อนขวา 5px",        lambda m: shift_mask(m, 5),       0.0),
        ("เลื่อนขวา 10px",       lambda m: shift_mask(m, 10),      0.0),
        ("เลื่อนขวา 20px",       lambda m: shift_mask(m, 20),      0.0),
        ("เลื่อนขวา 30px",       lambda m: shift_mask(m, 30),      0.0),
        ("ขยายเส้นหนา 5px",      lambda m: dilate_mask(m, 5),      0.0),
        ("ขยายเส้นหนา 9px",      lambda m: dilate_mask(m, 9),      0.0),
        ("หรอเส้นบาง 5px",       lambda m: erode_mask(m, 5),       0.0),
        ("หมุนเลน 1°",           lambda m: rotate_lanes(m, 1.0),   1.0),
        ("หมุนเลน 2°",           lambda m: rotate_lanes(m, 2.0),   2.0),
        ("หมุนเลน 3°",           lambda m: rotate_lanes(m, 3.0),   3.0),
        ("หมุนเลน 5°",           lambda m: rotate_lanes(m, 5.0),   5.0),
        ("หมุนเลน 10°",          lambda m: rotate_lanes(m, 10.0), 10.0),
        ("ลบเลนทิ้ง 1 เส้น",     lambda m: drop_one_lane(m, rng),  0.0),
    ]

    acc = {(nm, c): dict(iou=[], err=[], gt=0, m=0) for nm, _, _ in schemes for c in CATS}
    n_used = dict.fromkeys(CATS, 0)

    for cat in CATS:
        for rec in picks[cat]:
            gt = read_mask(rec)
            if gt is None:
                continue
            g = extract_lanes(gt)
            if not g:
                continue
            n_used[cat] += 1
            for nm, fn, _ in schemes:
                pm = fn(gt)
                res = slope_metrics_for_lanes(g, extract_lanes(pm))
                a = acc[(nm, cat)]
                a["iou"].append(mask_iou(gt, pm))
                a["gt"] += res["n_gt"]
                a["m"] += res["n_matched"]
                a["err"] += [d["err"] for d in res["pairs"]]

    long_rows = []
    for nm, _, expect in schemes:
        for cat in CATS:
            a = acc[(nm, cat)]
            e = np.array(a["err"]) if a["err"] else np.array([np.nan])
            long_rows.append(dict(
                perturbation=nm, category=cat, expected_angle=expect,
                n_img=n_used[cat],
                iou=float(np.mean(a["iou"])) if a["iou"] else float("nan"),
                slope_median=float(np.median(e)),
                lane_recall=(a["m"] / a["gt"]) if a["gt"] else float("nan"),
            ))
    df = pd.DataFrame(long_rows)

    if save_csv:
        os.makedirs("results/slope", exist_ok=True)
        out = "results/slope/slope_metric_by_category.csv"
        df.to_csv(out, index=False)
        print(f"📁 บันทึกไว้ที่: {out}")

    print(f"   ภาพที่ใช้จริงต่อหมวด: " + ", ".join(f"{c}={n_used[c]}" for c in CATS))
    return df, schemes


pert_df, PERT_SCHEMES = perturbation_by_category()


📁 บันทึกไว้ที่: results/slope/slope_metric_by_category.csv
   ภาพที่ใช้จริงต่อหมวด: normal=400, curve=400, night=400


In [11]:
def pivot_view(df, value, fmt="{:.3f}", label=""):
    """rows = การบิดเบือน, cols = รูปแบบถนน, บวกคอลัมน์ 'ช่วง' = max-min ระหว่างหมวด"""
    order = [nm for nm, _, _ in PERT_SCHEMES]
    p = df.pivot(index="perturbation", columns="category", values=value).reindex(order)[list(CATS)]
    p["ช่วง (max−min)"] = p[list(CATS)].max(axis=1) - p[list(CATS)].min(axis=1)
    p.index.name = label or value
    p.columns.name = None
    return p.map(lambda v: fmt.format(v) if pd.notna(v) else "-")


print("5.1  IoU — ทับกันแค่ไหน (สูง = ดี)\n")
view_iou = pivot_view(pert_df, "iou", "{:.3f}", "การบิดเบือน")
view_iou


5.1  IoU — ทับกันแค่ไหน (สูง = ดี)



,normal,curve,night,ช่วง (max−min)
การบิดเบือน,,,,
ไม่ทำอะไร (identity),1.000,1.000,1.000,0.000
เลื่อนขวา 5px,0.583,0.608,0.578,0.030
เลื่อนขวา 10px,0.313,0.350,0.307,0.043
เลื่อนขวา 20px,0.083,0.116,0.076,0.040
เลื่อนขวา 30px,0.033,0.043,0.020,0.023
ขยายเส้นหนา 5px,0.668,0.672,0.669,0.004
ขยายเส้นหนา 9px,0.502,0.507,0.504,0.005
หรอเส้นบาง 5px,0.508,0.515,0.510,0.008
หมุนเลน 1°,0.756,0.780,0.752,0.028


In [12]:
print("5.2  มุมคลาด median (°) — ทิศเพี้ยนแค่ไหน (ต่ำ = ดี)\n")
view_ang = pivot_view(pert_df, "slope_median", "{:.2f}", "การบิดเบือน")
view_ang


5.2  มุมคลาด median (°) — ทิศเพี้ยนแค่ไหน (ต่ำ = ดี)



,normal,curve,night,ช่วง (max−min)
การบิดเบือน,,,,
ไม่ทำอะไร (identity),0.00,0.00,0.00,0.00
เลื่อนขวา 5px,0.00,0.00,0.00,0.00
เลื่อนขวา 10px,0.00,0.00,0.00,0.00
เลื่อนขวา 20px,0.00,0.00,0.00,0.00
เลื่อนขวา 30px,0.00,0.00,0.00,0.00
ขยายเส้นหนา 5px,0.16,0.20,0.16,0.04
ขยายเส้นหนา 9px,0.50,0.61,0.41,0.20
หรอเส้นบาง 5px,0.08,0.14,0.07,0.07
หมุนเลน 1°,1.00,0.99,0.99,0.00


In [13]:
print("5.3  lane_recall — ยังจับคู่เส้นได้ครบไหม (สูง = ดี)\n")
view_rec = pivot_view(pert_df, "lane_recall", "{:.3f}", "การบิดเบือน")
view_rec


5.3  lane_recall — ยังจับคู่เส้นได้ครบไหม (สูง = ดี)



,normal,curve,night,ช่วง (max−min)
การบิดเบือน,,,,
ไม่ทำอะไร (identity),1.000,1.000,1.000,0.000
เลื่อนขวา 5px,1.000,1.000,1.000,0.000
เลื่อนขวา 10px,1.000,1.000,1.000,0.000
เลื่อนขวา 20px,1.000,1.000,1.000,0.000
เลื่อนขวา 30px,1.000,1.000,1.000,0.000
ขยายเส้นหนา 5px,0.855,0.846,0.853,0.009
ขยายเส้นหนา 9px,0.709,0.714,0.720,0.011
หรอเส้นบาง 5px,0.998,0.995,0.992,0.005
หมุนเลน 1°,0.997,0.984,0.992,0.012


In [14]:
# ==================== ด่านตรวจ ====================
ident = pert_df[pert_df["perturbation"] == "ไม่ทำอะไร (identity)"]
ok_ident = bool((np.abs(ident["iou"] - 1.0) < 1e-6).all() and (ident["slope_median"] < 1e-9).all())

rot = pert_df[pert_df["expected_angle"] > 0]
worst = float(np.max(np.abs(rot["slope_median"] - rot["expected_angle"])))
ok_rot = worst < 1.0

print("ด่านตรวจของตารางหัวข้อ 5")
print(f"  1) identity ทุกหมวด ได้ IoU 1.000 และ 0.00°   -> "
      f"{'✅ ผ่าน' if ok_ident else '❌ ไม่ผ่าน — มีบั๊ก อย่าอ่านผลต่อ'}")
print(f"  2) หมุน d° อ่านกลับได้ ~d° (คลาดมากสุด {worst:.2f}°) -> "
      f"{'✅ ผ่าน ฟังก์ชันบิดเบือนถูกต้อง' if ok_rot else '❌ ไม่ผ่าน — ฟังก์ชันบิดเบือนผิด'}")

# เทียบกับผลรวมทุกหมวดของ evaluate_metrics_comparison.ipynb เพื่อกันการสุ่มแบบสมดุลมีบั๊ก
_ref = "results/comparison/metric_perturbation_study.csv"
if os.path.exists(_ref):
    old = pd.read_csv(_ref).set_index("perturbation")
    new = pert_df.groupby("perturbation")[["iou", "slope_median"]].mean()
    cmp = pd.DataFrame({
        "IoU (ไฟล์นี้ เฉลี่ย 3 หมวด)": new["iou"].round(3),
        "IoU (ไฟล์เดิม ตามสัดส่วนจริง)": old["iou"].round(3),
        "มุม (ไฟล์นี้)": new["slope_median"].round(2),
        "มุม (ไฟล์เดิม)": old["slope_median"].round(2),
    }).reindex([nm for nm, _, _ in PERT_SCHEMES])
    print(f"\nเทียบกับ {_ref} — ต่างกันเล็กน้อยได้ เพราะไฟล์นี้ถ่วงหมวดเท่ากัน ส่วนไฟล์เดิมตามสัดส่วนจริง")
    display(cmp)
else:
    print(f"\n(ไม่มี {_ref} จึงข้ามการเทียบ — เป็นของแถม ไม่ใช่สิ่งที่ต้องมี")
    print(f" ผลทุกอย่างข้างบนสมบูรณ์อยู่แล้ว ไฟล์นั้นต้องใช้ GPU ส่วนไฟล์นี้ไม่ต้อง)")


ด่านตรวจของตารางหัวข้อ 5
  1) identity ทุกหมวด ได้ IoU 1.000 และ 0.00°   -> ✅ ผ่าน
  2) หมุน d° อ่านกลับได้ ~d° (คลาดมากสุด 0.09°) -> ✅ ผ่าน ฟังก์ชันบิดเบือนถูกต้อง

เทียบกับ results/comparison/metric_perturbation_study.csv — ต่างกันเล็กน้อยได้ เพราะไฟล์นี้ถ่วงหมวดเท่ากัน ส่วนไฟล์เดิมตามสัดส่วนจริง


,IoU (ไฟล์นี้ เฉลี่ย 3 หมวด),IoU (ไฟล์เดิม ตามสัดส่วนจริง),มุม (ไฟล์นี้),มุม (ไฟล์เดิม)
perturbation,,,,
ไม่ทำอะไร (identity),1.000,1.000,0.00,0.00
เลื่อนขวา 5px,0.590,0.587,0.00,0.00
เลื่อนขวา 10px,0.323,0.319,0.00,0.00
เลื่อนขวา 20px,0.092,0.088,0.00,0.00
เลื่อนขวา 30px,0.032,0.032,0.00,0.00
ขยายเส้นหนา 5px,0.670,0.669,0.17,0.18
ขยายเส้นหนา 9px,0.504,0.503,0.51,0.53
หรอเส้นบาง 5px,0.511,0.510,0.10,0.12
หมุนเลน 1°,0.763,0.759,0.99,0.99


## 6. ทำไมสามหมวดถึงต่างกันนิดหน่อย

ตารางหัวข้อ 5 จะออกมา **เกือบแบนราบ** โดยเฉพาะคอลัมน์มุมคลาด — **อันนี้ถูกต้องแล้ว ไม่ใช่บั๊ก**

เพราะเราบิดเบือนด้วยจำนวนเท่ากันทุกภาพ ความไวของ metric จึงเป็นสมบัติของ **รูปทรง mask**
ไม่ใช่ของ **ประเภทถนน** ผลลัพธ์ที่ได้คือ **การทดสอบควบคุม (control test)**:
พิสูจน์ว่า *ตัว metric เองไม่ลำเอียงตามรูปแบบถนน*

อันนี้สำคัญกว่าที่คิด เพราะไปค้ำข้อสรุปใน `evaluate_metrics_comparison.ipynb` ที่ว่า
*"ทางโค้ง IoU ขยับแค่ −1.2% แต่มุมคลาด +41%"* — ถ้าตัว metric เองลำเอียงตามหมวดอยู่แล้ว
ข้อสรุปนั้นจะเชื่อไม่ได้เลย ตารางนี้ปิดช่องโหว่นั้น

ส่วนต่างเล็กๆ ของ IoU ที่ยังเห็นอยู่ อธิบายได้ด้วยเรขาคณิตของ mask ล้วนๆ และมี **สองตัวขับ
คนละตัวกันตามชนิดการบิดเบือน** เซลล์ข้างล่างจะพิสูจน์ทั้งสองตัวด้วยการทำนายแล้วเทียบของจริง

| การบิดเบือน | ตัวขับ | เหตุผล |
| --- | --- | --- |
| **เลื่อน** | **ความหนาเส้น** `w = area / nrows` | แถบกว้าง `w` เลื่อนไป `d` ได้ IoU = `(w−d)/(w+d)` เส้นหนากว่าทนกว่า |
| **หมุน** | **ความยาวเส้น** | หมุนรอบ centroid ตัวเอง เส้นยาว = แขนโมเมนต์ยาว = ปลายเส้นเหวี่ยงออกไกล |

> **หมายเหตุ:** คำอธิบายที่ดูน่าเชื่อสองอันนี้ **ผิด** และถูกข้อมูลหักล้างไปแล้ว —
> *"เส้นต่อภาพน้อยกว่า → IoU สูงกว่า"* (night มีเส้นน้อยกว่า normal แต่ IoU ต่ำกว่า)
> และ *"เส้นห่างกันมากกว่า → IoU สูงกว่า"* (curve ระยะห่างน้อยที่สุด แต่ IoU สูงที่สุด)
> เซลล์ข้างล่างจึงตรวจลำดับด้วยโค้ด ไม่เขียนข้อสรุปตายตัว

In [15]:
# บริบทของ mask แต่ละหมวด — ใช้หาว่าส่วนต่าง IoU เล็กๆ มาจากอะไรกันแน่
picks_ctx = load_by_category(N_PER_CAT)

ctx_rows, W_BY_CAT, LEN_BY_CAT = [], {}, {}
for cat in CATS:
    n_lanes, widths, lengths, areas, gaps, badfit, n_img = [], [], [], [], [], [], 0
    for rec in picks_ctx[cat]:
        gt = read_mask(rec)
        if gt is None:
            continue
        g = extract_lanes(gt)
        if not g:
            continue
        n_img += 1
        n_lanes.append(len(g))
        for d in g:
            widths.append(d["area"] / d["nrows"])      # ความหนาเส้นแนวนอน (px)
            lengths.append(d["nrows"])                 # ความยาวเส้น (จำนวนแถว)
            areas.append(d["area"])
            badfit.append(1.0 if d["rms"] > SLOPE_MAX_RMS else 0.0)
        xs = sorted(d["xref"] for d in g)
        gaps += [b - a for a, b in zip(xs, xs[1:])]
    W_BY_CAT[cat] = np.array(widths)
    LEN_BY_CAT[cat] = np.array(lengths)
    ctx_rows.append((cat, n_img,
                     round(float(np.mean(n_lanes)), 2),
                     round(float(np.mean(widths)), 2),
                     int(np.median(lengths)),
                     int(np.median(areas)),
                     round(float(np.median(gaps)), 1) if gaps else float("nan"),
                     f"{np.mean(badfit) * 100:.1f}%"))

t6 = pd.DataFrame(ctx_rows, columns=[
    "ถนน", "ภาพ", "เส้น/ภาพ", "ความหนา w เฉลี่ย (px)", "ความยาว median (แถว)",
    "พื้นที่ median (px²)", "ระยะห่างเส้น median (px)",
    f"เส้นที่ rms > {SLOPE_MAX_RMS:.0f}px"]).set_index("ถนน")
print("บริบทของ mask แต่ละหมวด — สองคอลัมน์แรกหลัง 'เส้น/ภาพ' คือตัวขับจริง\n")
display(t6)


บริบทของ mask แต่ละหมวด — สองคอลัมน์แรกหลัง 'เส้น/ภาพ' คือตัวขับจริง



,ภาพ,เส้น/ภาพ,ความหนา w เฉลี่ย (px),ความยาว median (แถว),พื้นที่ median (px²),ระยะห่างเส้น median (px),เส้นที่ rms > 10px
ถนน,,,,,,,
normal,400,3.12,22.13,223,3778,346.7,8.9%
curve,400,2.73,24.82,149,2915,213.8,9.1%
night,400,2.88,21.50,233,3972,357.6,8.8%


In [16]:
# ---- ตัวขับ A: การเลื่อน ควรถูกกำหนดด้วย "ความหนาเส้น" ----
# แถบกว้าง w เลื่อนไป d: intersection = w-d, union = w+d  ->  IoU = (w-d)/(w+d)
# ทำนายจาก w "รายเส้น" แล้วเฉลี่ย ไม่ใช่เอาค่ากลางมาเข้าสูตร (ไม่เหมือนกัน)
def predict_shift_iou(w, d):
    return float(np.mean(np.clip((w - d) / (w + d), 0.0, 1.0)))


rows_a = []
for cat in CATS:
    w = W_BY_CAT[cat]
    r = {"ถนน": cat, "w เฉลี่ย": round(float(w.mean()), 2)}
    for d in (5, 10):
        meas = float(pert_df[(pert_df["perturbation"] == f"เลื่อนขวา {d}px")
                             & (pert_df["category"] == cat)]["iou"].iloc[0])
        pred = predict_shift_iou(w, d)
        r[f"ทำนาย {d}px"] = round(pred, 3)
        r[f"วัดได้ {d}px"] = round(meas, 3)
        r[f"พลาด {d}px"] = round(abs(pred - meas), 3)
    rows_a.append(r)
tA = pd.DataFrame(rows_a).set_index("ถนน")

worst_gap = float(tA[[c for c in tA.columns if c.startswith("พลาด")]].to_numpy().max())
order_w = [c for c in sorted(CATS, key=lambda c: -W_BY_CAT[c].mean())]
order_iou10 = list(pert_df[pert_df["perturbation"] == "เลื่อนขวา 10px"]
                   .sort_values("iou", ascending=False)["category"])
ok_a = (order_w == order_iou10) and worst_gap <= 0.02

print("A) การเลื่อน — ตัวขับคือความหนาเส้น\n")
display(tA)
print(f"   ลำดับความหนา   : {' > '.join(order_w)}")
print(f"   ลำดับ IoU 10px : {' > '.join(order_iou10)}")
print(f"   ทำนายพลาดมากสุด {worst_gap:.3f} (เกณฑ์ ≤ 0.02)")
print(f"   -> {'✓ ยืนยัน ความหนาเส้นอธิบายได้' if ok_a else '✗ ไม่ยืนยัน ต้องหาตัวขับอื่น'}")


A) การเลื่อน — ตัวขับคือความหนาเส้น



,w เฉลี่ย,ทำนาย 5px,วัดได้ 5px,พลาด 5px,ทำนาย 10px,วัดได้ 10px,พลาด 10px
ถนน,,,,,,,
normal,22.13,0.587,0.583,0.004,0.324,0.313,0.011
curve,24.82,0.607,0.608,0.001,0.355,0.350,0.005
night,21.50,0.585,0.578,0.007,0.321,0.307,0.014


   ลำดับความหนา   : curve > normal > night
   ลำดับ IoU 10px : curve > normal > night
   ทำนายพลาดมากสุด 0.014 (เกณฑ์ ≤ 0.02)
   -> ✓ ยืนยัน ความหนาเส้นอธิบายได้


In [17]:
# ---- ตัวขับ B: การหมุน ควรถูกกำหนดด้วย "ความยาวเส้น" (แขนโมเมนต์) ----
rows_b = []
for cat in CATS:
    meas = float(pert_df[(pert_df["perturbation"] == "หมุนเลน 5°")
                         & (pert_df["category"] == cat)]["iou"].iloc[0])
    rows_b.append((cat, int(np.median(LEN_BY_CAT[cat])), round(meas, 3)))
tB = pd.DataFrame(rows_b, columns=["ถนน", "ความยาว median (แถว)", "IoU หมุน 5°"]).set_index("ถนน")

order_len_short = [c for c, _, _ in sorted(rows_b, key=lambda x: x[1])]          # สั้น -> ยาว
order_iou_rot   = [c for c, _, _ in sorted(rows_b, key=lambda x: -x[2])]         # IoU สูง -> ต่ำ
ok_b = order_len_short == order_iou_rot

print("B) การหมุน — ตัวขับคือความยาวเส้น (ลำดับต้องกลับทางกัน)\n")
display(tB)
print(f"   เรียงจากเส้นสั้นสุด : {' < '.join(order_len_short)}")
print(f"   เรียงจาก IoU สูงสุด : {' > '.join(order_iou_rot)}")
print(f"   -> {'✓ ยืนยัน เส้นสั้นกว่าทนการหมุนได้ดีกว่า' if ok_b else '✗ ไม่ยืนยัน ต้องหาตัวขับอื่น'}")

# ---- ตัวขับที่ "ฟังดูน่าเชื่อแต่ผิด" ตรวจไว้กันเผลอเชื่อซ้ำ ----
n_lane = dict(zip(t6.index, t6["เส้น/ภาพ"]))
gap = dict(zip(t6.index, t6["ระยะห่างเส้น median (px)"]))
iou10 = {c: float(pert_df[(pert_df["perturbation"] == "เลื่อนขวา 10px")
                          & (pert_df["category"] == c)]["iou"].iloc[0]) for c in CATS}
wrong = [
    ("เส้นต่อภาพน้อยกว่า -> IoU สูงกว่า",
     sorted(CATS, key=lambda c: n_lane[c]) == sorted(CATS, key=lambda c: -iou10[c])),
    ("เส้นห่างกันมากกว่า -> IoU สูงกว่า",
     sorted(CATS, key=lambda c: -gap[c]) == sorted(CATS, key=lambda c: -iou10[c])),
]
print("\nคำอธิบายที่ฟังดูน่าเชื่อ แต่ข้อมูลไม่รองรับ (ตรวจซ้ำทุกครั้งที่รัน)")
for name, holds in wrong:
    print(f"   {name:40s} -> {'⚠ บังเอิญตรงรอบนี้' if holds else '✗ ไม่จริง'}")

print(f"\nสรุปเซลล์นี้: "
      f"{'✅ ตัวขับทั้งสองยืนยันได้' if (ok_a and ok_b) else '❌ ยังอธิบายไม่ได้ครบ อย่าเชื่อคำอธิบายในหัวข้อ 7'}")


B) การหมุน — ตัวขับคือความยาวเส้น (ลำดับต้องกลับทางกัน)



,ความยาว median (แถว),IoU หมุน 5°
ถนน,,
normal,223,0.241
curve,149,0.302
night,233,0.239


   เรียงจากเส้นสั้นสุด : curve < normal < night
   เรียงจาก IoU สูงสุด : curve > normal > night
   -> ✓ ยืนยัน เส้นสั้นกว่าทนการหมุนได้ดีกว่า

คำอธิบายที่ฟังดูน่าเชื่อ แต่ข้อมูลไม่รองรับ (ตรวจซ้ำทุกครั้งที่รัน)
   เส้นต่อภาพน้อยกว่า -> IoU สูงกว่า        -> ✗ ไม่จริง
   เส้นห่างกันมากกว่า -> IoU สูงกว่า        -> ✗ ไม่จริง

สรุปเซลล์นี้: ✅ ตัวขับทั้งสองยืนยันได้


## 7. สรุป

| คำถาม | คำตอบจากไฟล์นี้ |
| --- | --- |
| สูตรวัดความชันมีบั๊กไหม | ไม่มี — null test ได้ `0.0000°` / IoU `1.0000` / จับคู่ครบ **ทั้งสามหมวด** |
| ฟังก์ชันบิดเบือนเชื่อได้ไหม | ได้ — หมุน `d°` อ่านกลับมาได้ `~d°` ทุกหมวด |
| metric ลำเอียงตามรูปแบบถนนไหม | **ไม่ลำเอียง** — คอลัมน์มุมคลาดในตารางหัวข้อ 5 แบนราบ (ช่วง max−min ≤ 0.03° ทุกแถวหมุน) |
| แล้วส่วนต่าง IoU เล็กๆ มาจากไหน | **ความหนาเส้น** (ตอนเลื่อน) และ **ความยาวเส้น** (ตอนหมุน) ยืนยันด้วยสูตรทำนาย `(w−d)/(w+d)` ที่พลาดไม่เกิน 0.02 — **ไม่ใช่** จำนวนเส้นหรือระยะห่างเส้น ซึ่งทดสอบแล้วขัดกับข้อมูล |
| IoU กับความชัน ต่างกันตรงไหน | เลื่อนตำแหน่ง → IoU ตกหนักแต่มุมคลาด `0.00°` / หมุน → ตกทั้งคู่ / ลบเลนทิ้ง → เห็นเฉพาะ `lane_recall` |

**สิ่งที่ไฟล์นี้ไม่ได้ตอบ:** โมเดลเก่งแค่ไหน และบนถนนแบบไหน IoU กับความชันให้คำตอบขัดกัน
คำถามนั้นอยู่ที่ `evaluate_metrics_comparison.ipynb` ซึ่งใช้โมเดลจริงและต้องมี GPU

ทั้งสองไฟล์ต่อกันแบบนี้: ไฟล์นี้พิสูจน์ว่า **ไม้บรรทัดตรง** แล้วไฟล์นั้นค่อยเอาไม้บรรทัดไป **วัดโมเดล**